In [1]:
import json
import pandas as pd

BATCH_SIZE = 50
IS_DEBUG = True
DEBUG_QUESTION_SIZE = 10
# LLM_MODEL = "gpt-4.1-mini"
LLM_MODEL = "gpt-4.1"
DATASET_NAME = "hotpotQA"
KNOWLEDGE_GRAPH_PATH = 'outputs/knowledge_graphs'

## 1. Loading HotpotQA Dataset

In [2]:
train_hotpot_qa_path = r'C:\Users\Sudheera\Documents\Phd\LLMs_and_KGs\hotpotQA\hotpot_train_v1.1.json'
test_hotpot_qa_path = r'C:\Users\Sudheera\Documents\Phd\LLMs_and_KGs\hotpotQA\hotpot_test_v1.1.json'

In [3]:
with open(train_hotpot_qa_path, 'r', encoding='utf-8') as f:
    train_hotpot_qa_json = json.load(f)

with open(test_hotpot_qa_path, 'r', encoding='utf-8') as f:
    test_hotpot_qa_json = json.load(f)

In [4]:
train_hotpot_qa_json[0]

{'supporting_facts': [["Arthur's Magazine", 0], ['First for Women', 0]],
 'level': 'medium',
 'question': "Which magazine was started first Arthur's Magazine or First for Women?",
 'context': [['Radio City (Indian radio station)',
   ["Radio City is India's first private FM radio station and was started on 3 July 2001.",
    ' It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003).',
    ' It plays Hindi, English and regional songs.',
    ' It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007.',
    ' Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, and other music-related features.',
    ' The Radio station currently plays a mix of Hindi and Regional music.',
    ' Abraham Thomas is the CEO of the company.']]

In [5]:
test_hotpot_qa_json[0]

{'_id': '5a8b57f25542995d1e6f1371',
 'answer': 'yes',
 'question': 'Were Scott Derrickson and Ed Wood of the same nationality?',
 'supporting_facts': [['Scott Derrickson', 0], ['Ed Wood', 0]],
 'context': [['Adam Collis',
   ['Adam Collis is an American filmmaker and actor.',
    ' He attended the Duke University from 1986 to 1990 and the University of California, Los Angeles from 2007 to 2010.',
    ' He also studied cinema at the University of Southern California from 1991 to 1997.',
    ' Collis first work was the assistant director for the Scott Derrickson\'s short "Love in the Ruins" (1995).',
    ' In 1998, he played "Crankshaft" in Eric Koyanagi\'s "Hundred Percent".']],
  ['Ed Wood (film)',
   ['Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.',
    " The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lug

In [6]:
for item in train_hotpot_qa_json[0]['context']:
    print(f"Title: {item[0]}")
    print(f"Paragraph: {item[1:]}")
    print("-" * 50)

Title: Radio City (Indian radio station)
Paragraph: [["Radio City is India's first private FM radio station and was started on 3 July 2001.", ' It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003).', ' It plays Hindi, English and regional songs.', ' It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007.', ' Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, and other music-related features.', ' The Radio station currently plays a mix of Hindi and Regional music.', ' Abraham Thomas is the CEO of the company.']]
--------------------------------------------------
Title: History of Albanian football
Paragraph: [['Football in Albania existed before the Albanian Football Federation (FSHF) was created.', " This was ev

In [7]:
def divide_and_batch(train_json, batch_size):
    batch_context = []
    batch_question = []
    batch_answer = []

    for i in range(0, len(train_json), batch_size):
        batch = train_json[i:i + batch_size]
        context = []
        question = []
        answer = []

        for item in batch:
            context_text = ""
            j = 1
            for ctx in item['context']:
                context_text += f"Title {j} : {ctx[0]} \nParagraph {j} : {''.join(ctx[1])}\n"
                j += 1
            context.append(context_text)
            question.append(item['question'])
            answer.append(item['answer'])

        batch_context.append(context)
        batch_question.append(question)
        batch_answer.append(answer)

    return batch_context, batch_question, batch_answer

In [8]:
if IS_DEBUG:
    train_hotpot_qa_json = train_hotpot_qa_json[:DEBUG_QUESTION_SIZE]
    test_hotpot_qa_json = test_hotpot_qa_json[:DEBUG_QUESTION_SIZE]
    batch_context, batch_question, batch_answer = divide_and_batch(train_hotpot_qa_json, BATCH_SIZE)
else:
    batch_context, batch_question, batch_answer = divide_and_batch(test_hotpot_qa_json, BATCH_SIZE)

In [9]:
for i in range(len(batch_context[0])):
    print(f"Batch {i + 1}:")
    print("Context:", batch_context[0][i])
    print("Question:", batch_question[0][i])
    print("Answer:", batch_answer[0][i])
    print("-" * 50)
    if i == 9:  # Limit to 3 batches for brevity
        break

Batch 1:
Context: Title 1 : Radio City (Indian radio station) 
Paragraph 1 : Radio City is India's first private FM radio station and was started on 3 July 2001. It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003). It plays Hindi, English and regional songs. It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007. Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, and other music-related features. The Radio station currently plays a mix of Hindi and Regional music. Abraham Thomas is the CEO of the company.
Title 2 : History of Albanian football 
Paragraph 2 : Football in Albania existed before the Albanian Football Federation (FSHF) was created. This was evidenced by the team's registration at the Balkan Cup tou

In [10]:
del train_hotpot_qa_json
del test_hotpot_qa_json

## 2. Getting Triples from Context and Question

In [11]:
from dotenv import load_dotenv
import os

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

In [12]:
from langchain_openai import ChatOpenAI
from langchain_core.globals import set_debug, set_verbose, set_llm_cache
from langchain_community.cache import InMemoryCache
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

set_debug(False)
set_verbose(False)
set_llm_cache(InMemoryCache())

llm_model = ChatOpenAI(model=LLM_MODEL, api_key=openai_api_key)

In [13]:
system_msg = """
<role>
    You are an expert in natural language understanding and knowledge graph construction. Your task is to extract structured relationship triples from a context set (Titles and Paragraphs), including both explicit and logically inferable facts.
</role>

<behavior>
    <rule name="Entity Extraction">
        Identify and extract unique entities from Titles and Paragraphs. Use canonical forms for names. Capture entities including people, organizations, locations, publications, dates, and events.
    </rule>
    <rule name="Relationship Extraction">
        Extract explicit and clearly implied relationships between entities in the form of (subject, relation, object) triples.
        This includes:
        - Temporal relations such as publication or operational time spans (e.g., "published during", "active from", "merged in").
        - Authorship, editorial, or contribution roles (e.g., "edited by", "featured work by").
        - Event-based or structural relations (e.g., "merged into", "founded by", "took over").
    </rule>
    <rule name="Completeness">
        Extract **as many valid and informative triples as possible**. Include relationships that are inferred logically but unambiguous in context (e.g., date ranges, geographic associations, contributor roles).
    </rule>
    <rule name="Formatting and Output">
        Output must contain two sections, in order:
        1. ### CONTEXT_REASONING — step-by-step reasoning describing how entities and relationships were selected or inferred.
        2. ### CONTEXT_TRIPLES — list all extracted triples, each on its own line in the format (entity_1, relationship, entity_2)
        Do not add explanations, numbering, or extra text outside of these sections.
    </rule>
</behavior>

<format>
1. Carefully read the <context> section.
2. Begin with this heading:
   ### CONTEXT_REASONING
   Then, write concise reasoning steps describing how entities and relationships were selected and interpreted.
3. After reasoning, print this heading:
   ### CONTEXT_TRIPLES
   Then list each extracted triple on a new line in the format (entity_1, relationship, entity_2).
4. Do not include any explanations, text, or comments after the last triple.
</format>
"""

human_msg = """
<context>
{context}
</context>
"""

prompt = ChatPromptTemplate([("system", system_msg), ("human", human_msg)])

In [14]:
chain = prompt | llm_model | StrOutputParser()

In [15]:
batch_input_list = []
main_df = pd.DataFrame()

for i in range(len(batch_context)):
    batch_num_list = [i] * len(batch_context[i])
    question_num_list = list(range(1, len(batch_context[i]) + 1))
    context_list = []
    question_list = []
    answers_list = []

    batch_input_item = []
    j = 0
    for batch in range(len(batch_context[i])):
        context = batch_context[i][batch]
        batch_input_item.append({
            "context": context
        })
        context_list.append(context)
        question_list.append(batch_question[i][j])
        answers_list.append(batch_answer[i][j])
        j += 1
    batch_input_list.append(batch_input_item)
    df = pd.DataFrame({
        "batch_num": batch_num_list,
        "question_num": question_num_list,
        "context": context_list,
        "question": question_list,
        "answer": answers_list
    })
    main_df = pd.concat([main_df, df], ignore_index=True)

for i, batch in enumerate(batch_input_list):
    print(f"Batch {i + 1}:")
    for item in batch:
        print("Context:", item['context'])
        print("-" * 50)

Batch 1:
Context: Title 1 : Radio City (Indian radio station) 
Paragraph 1 : Radio City is India's first private FM radio station and was started on 3 July 2001. It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003). It plays Hindi, English and regional songs. It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007. Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, and other music-related features. The Radio station currently plays a mix of Hindi and Regional music. Abraham Thomas is the CEO of the company.
Title 2 : History of Albanian football 
Paragraph 2 : Football in Albania existed before the Albanian Football Federation (FSHF) was created. This was evidenced by the team's registration at the Balkan Cup tou

In [16]:
main_df

,batch_num,question_num,context,question,answer
0,0,1,Title 1 : Radio City (Indian radio station) \n...,Which magazine was started first Arthur's Maga...,Arthur's Magazine
1,0,2,Title 1 : Ritz-Carlton Jakarta \nParagraph 1 :...,The Oberoi family is part of a hotel company t...,Delhi
2,0,3,Title 1 : Lisa Simpson \nParagraph 1 : Lisa Ma...,Musician and satirist Allie Goertz wrote a son...,President Richard Nixon
3,0,4,"Title 1 : Moloch: or, This Gentile World \nPar...",What nationality was James Henry Miller's wife?,American
4,0,5,Title 1 : Cadmium chloride \nParagraph 1 : Cad...,Cadmium Chloride is slightly soluble in this c...,alcohol
5,0,6,Title 1 : Li Na \nParagraph 1 : Li Na (; ; bor...,Which tennis player won more Grand Slam titles...,Jonathan Stark
6,0,7,"Title 1 : India \nParagraph 1 : India, officia...",Which genus of moth in the world's seventh-lar...,Crambidae
7,0,8,Title 1 : Verano de Escándalo (1998) \nParagra...,Who was once considered the best kick boxer in...,Badr Hari
8,0,9,Title 1 : House of Anubis \nParagraph 1 : Hous...,"The Dutch-Belgian television series that ""Hous...",2006
9,0,10,Title 1 : Mount Panorama Circuit \nParagraph 1...,What is the length of the track where the 2013...,6.213 km long


In [17]:
graphs_str_batch = []

for i, batch in enumerate(batch_input_list):
    print(f"Batch {i + 1} ... processing {len(batch)} items")
    graphs_str_list = chain.batch(batch)
    graphs_str_batch.append(graphs_str_list)

Batch 1 ... processing 10 items


In [18]:
for i, graphs_str_list in enumerate(graphs_str_batch):
    print(f"Batch {i + 1} ... processed {len(graphs_str_list)} items")
    for j, graph_str in enumerate(graphs_str_list):
        print(f"Item {j + 1}:")
        print(graph_str)
        print("-" * 50)

Batch 1 ... processed 10 items
Item 1:
### CONTEXT_REASONING
Entities were extracted by identifying names of organizations, people, places, events, publications, dates, and notable subject nouns across all paragraphs. Relationships were derived from verbs and logically implied facts, such as operational spans, launch events, founder roles, published in or during time periods, and mergers. For instance, “Radio City” entities connect to specific cities and dates as stations launched there. Authorship and editorial relationships were found for magazines, with contributor work inferred where artists’ works were featured. Event unions (such as fire mergers), achievements (such as championship wins), and specifics about organizations (like target demographic or geographic location) were also meticulously extracted. For entities mentioned with time qualifiers (e.g., “oldest”, “first”), both temporal and distinction-based relationships were considered.

### CONTEXT_TRIPLES
(Radio City, is, rad

In [19]:
graphs_str_df = pd.DataFrame({
    "batch_num": [],
    "question_num": [],
    "graphs_str": []
})

for i, graphs_str_list in enumerate(graphs_str_batch):
    batch_num_list = [i] * len(graphs_str_list)
    question_num_list = list(range(1, len(graphs_str_list) + 1))
    graphs_str_df = pd.concat([graphs_str_df, pd.DataFrame({
        "batch_num": batch_num_list,
        "question_num": question_num_list,
        "graphs_str": graphs_str_list
    })], ignore_index=True)

del graphs_str_batch
del graphs_str_list
del batch_input_list
del batch_context
del batch_question
del batch_answer

graphs_str_df

,batch_num,question_num,graphs_str
0,0.0,1.0,### CONTEXT_REASONING\nEntities were extracted...
1,0.0,2.0,### CONTEXT_REASONING\nEntities were extracted...
2,0.0,3.0,### CONTEXT_REASONING\nEntities are extracted ...
3,0.0,4.0,### CONTEXT_REASONING\nEntities were extracted...
4,0.0,5.0,### CONTEXT_REASONING\nEntities were extracted...
5,0.0,6.0,### CONTEXT_REASONING\nEntities were identifie...
6,0.0,7.0,### CONTEXT_REASONING\nEntities were extracted...
7,0.0,8.0,"### CONTEXT_REASONING\nFirst, I identified uni..."
8,0.0,9.0,### CONTEXT_REASONING\nI identified television...
9,0.0,10.0,"### CONTEXT_REASONING\nFirst, I identified maj..."


## 3. Parsing the Output and Building KGs

In [20]:
import re
from ast import literal_eval


def sanitize_str(name):
    """
    Sanitize a name by removing extra spaces, apostrophes, and ensuring proper formatting.
    """
    # Replace apostrophes with empty string
    name = name.replace("'s", "s")
    name = name.replace("'", "")

    # Replace spaces with underscores
    name = name.replace(" ", "_")
    name = name.replace("–", "_")

    # Remove any other problematic characters
    name = re.sub(r'[^\w\.\-_]', '', name)

    # Prefix if starts with digit
    if re.match(r"^\d", name):
        name = f"n{name}"

    # If empty string, return something safe
    if not name:
        name = "unknown"

    return name


def sanitize_entities_and_relations(triple):
    """
    Sanitize entities and relations in a triple by removing extra spaces and ensuring proper formatting.
    """
    return (
        sanitize_str(triple[0].strip().lower()),  # Subject
        sanitize_str(triple[1].strip().lower()),  # Relation
        sanitize_str(triple[2].strip().lower())  # Object
    )


def parse_custom_triples(triples_str):
    triples = []
    for line in triples_str.strip().splitlines():
        line = line.strip()
        if not line:
            continue
        # Remove enclosing parentheses
        if line.startswith('(') and line.endswith(')'):
            line = line[1:-1]
        # Now, split ONLY on the first two commas
        parts = []
        remaining = line
        for _ in range(2):
            # Find the first comma
            idx = remaining.find(',')
            if idx == -1:
                break
            parts.append(remaining[:idx].strip())
            remaining = remaining[idx + 1:].strip()
        parts.append(remaining)
        if len(parts) == 3:
            triples.append(sanitize_entities_and_relations(tuple(parts)))
    return triples


def extract_sections(text, reasoning_pat, triples_pat):
    # Extract sections using regex
    extracted_reasoning = re.search(reasoning_pat, text, re.DOTALL)
    extracted_triples = re.search(triples_pat, text, re.DOTALL)

    # Clean reasoning sections
    reasoning_str = extracted_reasoning.group(1).strip() if extracted_reasoning else ""

    # Use the custom parser for triples
    triples_list = parse_custom_triples(extracted_triples.group(1)) if extracted_triples else []

    return reasoning_str, triples_list


def extract_context_to_columns(row):
    # Regex patterns for section headers
    context_reasoning_pat = r'### CONTEXT_REASONING\s*(.*?)\s*### CONTEXT_TRIPLES'
    context_triples_pat = r'### CONTEXT_TRIPLES\s*(.*)'

    context_reasoning, context_triples = extract_sections(row['graphs_str'], context_reasoning_pat, context_triples_pat)
    return pd.Series([context_reasoning, context_triples],
                     index=['context_reasoning', 'context_triples'])

In [21]:
graphs_str_df[['context_reasoning', 'context_triples']] = graphs_str_df.apply(extract_context_to_columns, axis=1)
graphs_str_df

,batch_num,question_num,graphs_str,context_reasoning,context_triples
0,0.0,1.0,### CONTEXT_REASONING\nEntities were extracted...,Entities were extracted by identifying names o...,"[(radio_city, is, radio_station), (radio_city,..."
1,0.0,2.0,### CONTEXT_REASONING\nEntities were extracted...,Entities were extracted based on clear mention...,"[(ritz-carlton_jakarta, is_a, hotel), (ritz-ca..."
2,0.0,3.0,### CONTEXT_REASONING\nEntities are extracted ...,Entities are extracted by identifying unique n...,"[(lisa_simpson, is_fictional_character_in, the..."
3,0.0,4.0,### CONTEXT_REASONING\nEntities were extracted...,Entities were extracted from titles and paragr...,"[(moloch_or, this_gentile_world, written_by_he..."
4,0.0,5.0,### CONTEXT_REASONING\nEntities were extracted...,Entities were extracted from each title and pa...,"[(cadmium_chloride, is_a, chemical_compound), ..."
5,0.0,6.0,### CONTEXT_REASONING\nEntities were identifie...,Entities were identified as prominent individu...,"[(li_na, born_on, n26_february_1982), (li_na, ..."
6,0.0,7.0,### CONTEXT_REASONING\nEntities were extracted...,Entities were extracted by identifying country...,"[(india, official_name, republic_of_india), (i..."
7,0.0,8.0,"### CONTEXT_REASONING\nFirst, I identified uni...","First, I identified unique entities from each ...","[(verano_de_escándalo_1998, held_on, september..."
8,0.0,9.0,### CONTEXT_REASONING\nI identified television...,"I identified television series, people, organi...","[(house_of_anubis, is_a, television_series), (..."
9,0.0,10.0,"### CONTEXT_REASONING\nFirst, I identified maj...","First, I identified major entities: Mount Pano...","[(mount_panorama_circuit, located_in, bathurst..."


In [22]:
main_df = main_df.merge(graphs_str_df, on=['batch_num', 'question_num'], how='left')
# del graphs_str_df
main_df

,batch_num,question_num,context,question,answer,graphs_str,context_reasoning,context_triples
0,0,1,Title 1 : Radio City (Indian radio station) \n...,Which magazine was started first Arthur's Maga...,Arthur's Magazine,### CONTEXT_REASONING\nEntities were extracted...,Entities were extracted by identifying names o...,"[(radio_city, is, radio_station), (radio_city,..."
1,0,2,Title 1 : Ritz-Carlton Jakarta \nParagraph 1 :...,The Oberoi family is part of a hotel company t...,Delhi,### CONTEXT_REASONING\nEntities were extracted...,Entities were extracted based on clear mention...,"[(ritz-carlton_jakarta, is_a, hotel), (ritz-ca..."
2,0,3,Title 1 : Lisa Simpson \nParagraph 1 : Lisa Ma...,Musician and satirist Allie Goertz wrote a son...,President Richard Nixon,### CONTEXT_REASONING\nEntities are extracted ...,Entities are extracted by identifying unique n...,"[(lisa_simpson, is_fictional_character_in, the..."
3,0,4,"Title 1 : Moloch: or, This Gentile World \nPar...",What nationality was James Henry Miller's wife?,American,### CONTEXT_REASONING\nEntities were extracted...,Entities were extracted from titles and paragr...,"[(moloch_or, this_gentile_world, written_by_he..."
4,0,5,Title 1 : Cadmium chloride \nParagraph 1 : Cad...,Cadmium Chloride is slightly soluble in this c...,alcohol,### CONTEXT_REASONING\nEntities were extracted...,Entities were extracted from each title and pa...,"[(cadmium_chloride, is_a, chemical_compound), ..."
5,0,6,Title 1 : Li Na \nParagraph 1 : Li Na (; ; bor...,Which tennis player won more Grand Slam titles...,Jonathan Stark,### CONTEXT_REASONING\nEntities were identifie...,Entities were identified as prominent individu...,"[(li_na, born_on, n26_february_1982), (li_na, ..."
6,0,7,"Title 1 : India \nParagraph 1 : India, officia...",Which genus of moth in the world's seventh-lar...,Crambidae,### CONTEXT_REASONING\nEntities were extracted...,Entities were extracted by identifying country...,"[(india, official_name, republic_of_india), (i..."
7,0,8,Title 1 : Verano de Escándalo (1998) \nParagra...,Who was once considered the best kick boxer in...,Badr Hari,"### CONTEXT_REASONING\nFirst, I identified uni...","First, I identified unique entities from each ...","[(verano_de_escándalo_1998, held_on, september..."
8,0,9,Title 1 : House of Anubis \nParagraph 1 : Hous...,"The Dutch-Belgian television series that ""Hous...",2006,### CONTEXT_REASONING\nI identified television...,"I identified television series, people, organi...","[(house_of_anubis, is_a, television_series), (..."
9,0,10,Title 1 : Mount Panorama Circuit \nParagraph 1...,What is the length of the track where the 2013...,6.213 km long,"### CONTEXT_REASONING\nFirst, I identified maj...","First, I identified major entities: Mount Pano...","[(mount_panorama_circuit, located_in, bathurst..."


In [23]:
# Loop through each row. Save the context triples as OWL KG files
from rdflib import Graph, Namespace, RDF, RDFS, OWL, URIRef
import os


def save_kg_from_triples(triples, batch_num, question_num):
    g = Graph()
    EX = Namespace("http://example.org/")

    # Add triples to the graph
    for subject, relation, obj in triples:
        subject_uri = URIRef(EX[subject])
        object_uri = URIRef(EX[obj])
        g.add((subject_uri, URIRef(EX[relation]), object_uri))

    # Define the file path
    file_path = f"{KNOWLEDGE_GRAPH_PATH}/{DATASET_NAME}_ontology_b{batch_num}_q{question_num}.rdf"

    # Save the graph in OWL format
    g.serialize(destination=file_path, format='xml')
    print(f"Saved KG for batch {batch_num}, question {question_num} to {file_path}")


for index, row in main_df.iterrows():
    batch_num = row['batch_num']
    question_num = row['question_num']
    context_triples = row['context_triples']

    if context_triples:  # Only save if there are context triples
        save_kg_from_triples(context_triples, batch_num, question_num)

Saved KG for batch 0, question 1 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q1.rdf
Saved KG for batch 0, question 2 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q2.rdf
Saved KG for batch 0, question 3 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q3.rdf
Saved KG for batch 0, question 4 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q4.rdf
Saved KG for batch 0, question 5 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q5.rdf
Saved KG for batch 0, question 6 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q6.rdf
Saved KG for batch 0, question 7 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q7.rdf
Saved KG for batch 0, question 8 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q8.rdf
Saved KG for batch 0, question 9 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q9.rdf
Saved KG for batch 0, question 10 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q10.rdf


## 4. Extracting Entities from Questions

In [24]:
batch_input_list = []

for i in range(0, main_df.shape[0], BATCH_SIZE):
    batch_df = main_df.iloc[i:i + BATCH_SIZE]
    batch_input_item = []

    for i, (index, row) in enumerate(batch_df.iterrows()):
        batch_input_item.append({
            "question": row['question']
        })
    batch_input_list.append(batch_input_item)

for i, batch in enumerate(batch_input_list):
    print(f"Batch {i + 1}:")
    for item in batch:
        print("Question:", item['question'])
        print("-" * 50)

Batch 1:
Question: Which magazine was started first Arthur's Magazine or First for Women?
--------------------------------------------------
Question: The Oberoi family is part of a hotel company that has a head office in what city?
--------------------------------------------------
Question: Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?
--------------------------------------------------
Question:  What nationality was James Henry Miller's wife?
--------------------------------------------------
Question: Cadmium Chloride is slightly soluble in this chemical, it is also called what?
--------------------------------------------------
Question: Which tennis player won more Grand Slam titles, Henri Leconte or Jonathan Stark?
--------------------------------------------------
Question: Which genus of moth in the world's seventh-largest country contains only one species?
---------------------------------------

In [25]:
system_msg = """
<role>
    You are an expert in natural language understanding and knowledge graph construction. Your task is to convert a provided question into a set of question triples, using simple variable names (such as x, y, z, i, j, k) for unknown entities or values to be inferred.
</role>

<behavior>
    <rule name="Entity Extraction">
        Identify and extract unique entities. Use canonical names.
    </rule>
    <rule name="Relationship Extraction">
        Extract explicit or clear implicit relationships between entities as (subject, relation, object) triples.
    </rule>
    <rule name="Temporal and Comparative Relations">
        Use relations like "started in", "founded by", "wrote a song about", "is named after", etc., matching the logic and semantics in the context or question.
    </rule>
    <rule name="Variable Placeholders for Unknowns">
        For each unknown or answer to be found in the question, use a simple English variable (x, y, z, i, j, k, etc.) in place of the answer within the triples. Each distinct unknown in the question gets a unique variable.
    </rule>
    <rule name="Question Triple Decomposition">
        For each question, decompose it into one or more triples, using variable placeholders for any unknowns to be inferred from the context.
        Examples:
        - "Which magazine was started first Arthur's Magazine or First for Women?"
           (Arthur's Magazine, started in, x)
           (First for Women, started in, y)
        - "The Oberoi family is part of a hotel company that has a head office in what city?"
           (Oberoi family, is part of, x)
           (x, has head office in, y)
    </rule>
    <rule name="Formatting and Output">
        Output must contain four sections, in order:
        1. ### QUESTION_REASONING — a step-by-step reasoning of how the question was decomposed into question triples
        2. ### QUESTIONS_TRIPLES — list the question triples, each on its own line. each on its own line in the format (entity_1, relationship, entity_2)
        Do not add explanations, numbering, or extra text.
    </rule>
</behavior>

<format>
1. Carefully read the <question> section.
2. Before extracting question triples, explain step by step how you break down the question and assign variable placeholders. Print this heading:
   ### QUESTION_REASONING
   Then write your reasoning in clear, concise steps.
3. After reasoning, print the question triples. Print this heading:
   ### QUESTIONS_TRIPLES
   List each question triple on a new line, using variable placeholders (x, y, z, etc.) for unknowns.
4. After the last question triple, end with no extra text or parentheses.
</format>
"""

human_msg = """
<question>
{question}
</question>
"""

prompt = ChatPromptTemplate([("system", system_msg), ("human", human_msg)])

In [26]:
chain = prompt | llm_model | StrOutputParser()

In [27]:
question_str_batch = []

for i, batch in enumerate(batch_input_list):
    print(f"Batch {i + 1} ... processing {len(batch)} items")
    question_str_list = chain.batch(batch)
    question_str_batch.append(question_str_list)

Batch 1 ... processing 10 items


In [28]:
for i, question_str_list in enumerate(question_str_batch):
    print(f"Batch {i + 1} ... processed {len(question_str_list)} items")
    for j, graph_str in enumerate(question_str_list):
        print(f"Item {j + 1}:")
        print(graph_str)
        print("-" * 50)

Batch 1 ... processed 10 items
Item 1:
### QUESTION_REASONING
I identified two entities in the question: "Arthur's Magazine" and "First for Women". The question is asking to compare the starting dates of both magazines to determine which was started first. The unknowns here are the dates each magazine started, so I assign variables to these dates: x for Arthur's Magazine and y for First for Women.

### QUESTIONS_TRIPLES
(Arthur's Magazine, started in, x)
(First for Women, started in, y)
--------------------------------------------------
Item 2:
### QUESTION_REASONING
The question is asking for the city where the head office of a certain hotel company is located. The company in question is one that the Oberoi family is part of. There are two key unknowns: the name of the hotel company (represented by variable x), and the city where its head office is located (represented by variable y). First, we relate the Oberoi family to the hotel company using "is part of". Next, we relate the hotel

In [29]:
questions_str_df = pd.DataFrame({
    "batch_num": [],
    "question_num": [],
    "questions_str": []
})

for i, graphs_str_list in enumerate(question_str_batch):
    batch_num_list = [i] * len(graphs_str_list)
    question_num_list = list(range(1, len(graphs_str_list) + 1))
    questions_str_df = pd.concat([questions_str_df, pd.DataFrame({
        "batch_num": batch_num_list,
        "question_num": question_num_list,
        "questions_str": graphs_str_list
    })], ignore_index=True)

questions_str_df

,batch_num,question_num,questions_str
0,0.0,1.0,### QUESTION_REASONING\nI identified two entit...
1,0.0,2.0,### QUESTION_REASONING\nThe question is asking...
2,0.0,3.0,"### QUESTION_REASONING\nFirst, identify the ma..."
3,0.0,4.0,### QUESTION_REASONING\nThe question asks for ...
4,0.0,5.0,"### QUESTION_REASONING\nFirst, identify entiti..."
5,0.0,6.0,"### QUESTION_REASONING\nFirst, identify the tw..."
6,0.0,7.0,### QUESTION_REASONING\n1. The question asks f...
7,0.0,8.0,"### QUESTION_REASONING\nFirst, I identify the ..."
8,0.0,9.0,"### QUESTION_REASONING\nFirst, identify that ""..."
9,0.0,10.0,"### QUESTION_REASONING\nFirst, identify the ma..."


In [30]:
def extract_question_to_columns(row):
    # Regex patterns for section headers
    questions_reasoning_pat = r'### QUESTION_REASONING\s*(.*?)\s*### QUESTIONS_TRIPLES'
    questions_triples_pat = r'### QUESTIONS_TRIPLES\s*(.*)'

    questions_reasoning, questions_triples = extract_sections(row['questions_str'], questions_reasoning_pat,
                                                              questions_triples_pat)
    return pd.Series([questions_reasoning, questions_triples],
                     index=['question_reasoning', 'question_triples'])


questions_str_df[['question_reasoning', 'question_triples']] = questions_str_df.apply(extract_question_to_columns,
                                                                                      axis=1)
questions_str_df

,batch_num,question_num,questions_str,question_reasoning,question_triples
0,0.0,1.0,### QUESTION_REASONING\nI identified two entit...,"I identified two entities in the question: ""Ar...","[(arthurs_magazine, started_in, x), (first_for..."
1,0.0,2.0,### QUESTION_REASONING\nThe question is asking...,The question is asking for the city where the ...,"[(oberoi_family, is_part_of, x), (x, has_head_..."
2,0.0,3.0,"### QUESTION_REASONING\nFirst, identify the ma...","First, identify the main entities: Allie Goert...","[(allie_goertz, wrote_a_song_about, milhouse),..."
3,0.0,4.0,### QUESTION_REASONING\nThe question asks for ...,The question asks for the nationality of James...,"[(james_henry_miller, has_wife, x), (x, has_na..."
4,0.0,5.0,"### QUESTION_REASONING\nFirst, identify entiti...","First, identify entities: ""Cadmium Chloride"" a...","[(cadmium_chloride, is_slightly_soluble_in, x)..."
5,0.0,6.0,"### QUESTION_REASONING\nFirst, identify the tw...","First, identify the two entities being compare...","[(henri_leconte, won_number_of_grand_slam_titl..."
6,0.0,7.0,### QUESTION_REASONING\n1. The question asks f...,1. The question asks for a genus of moth (unkn...,"[(x, is_a_genus_of, moth), (x, located_in, y),..."
7,0.0,8.0,"### QUESTION_REASONING\nFirst, I identify the ...","First, I identify the main subject of the ques...","[(x, was_considered_the_best_in, kick_boxing),..."
8,0.0,9.0,"### QUESTION_REASONING\nFirst, identify that ""...","First, identify that ""House of Anubis"" is base...","[(house_of_anubis, is_based_on, x), (x, first_..."
9,0.0,10.0,"### QUESTION_REASONING\nFirst, identify the ma...","First, identify the main event: ""2013 Liqui Mo...","[(n2013_liqui_moly_bathurst_12_hour, was_stage..."


In [31]:
main_df = main_df.merge(questions_str_df, on=['batch_num', 'question_num'], how='left')
# del graphs_str_df
main_df

,batch_num,question_num,context,question,answer,graphs_str,context_reasoning,context_triples,questions_str,question_reasoning,question_triples
0,0,1,Title 1 : Radio City (Indian radio station) \n...,Which magazine was started first Arthur's Maga...,Arthur's Magazine,### CONTEXT_REASONING\nEntities were extracted...,Entities were extracted by identifying names o...,"[(radio_city, is, radio_station), (radio_city,...",### QUESTION_REASONING\nI identified two entit...,"I identified two entities in the question: ""Ar...","[(arthurs_magazine, started_in, x), (first_for..."
1,0,2,Title 1 : Ritz-Carlton Jakarta \nParagraph 1 :...,The Oberoi family is part of a hotel company t...,Delhi,### CONTEXT_REASONING\nEntities were extracted...,Entities were extracted based on clear mention...,"[(ritz-carlton_jakarta, is_a, hotel), (ritz-ca...",### QUESTION_REASONING\nThe question is asking...,The question is asking for the city where the ...,"[(oberoi_family, is_part_of, x), (x, has_head_..."
2,0,3,Title 1 : Lisa Simpson \nParagraph 1 : Lisa Ma...,Musician and satirist Allie Goertz wrote a son...,President Richard Nixon,### CONTEXT_REASONING\nEntities are extracted ...,Entities are extracted by identifying unique n...,"[(lisa_simpson, is_fictional_character_in, the...","### QUESTION_REASONING\nFirst, identify the ma...","First, identify the main entities: Allie Goert...","[(allie_goertz, wrote_a_song_about, milhouse),..."
3,0,4,"Title 1 : Moloch: or, This Gentile World \nPar...",What nationality was James Henry Miller's wife?,American,### CONTEXT_REASONING\nEntities were extracted...,Entities were extracted from titles and paragr...,"[(moloch_or, this_gentile_world, written_by_he...",### QUESTION_REASONING\nThe question asks for ...,The question asks for the nationality of James...,"[(james_henry_miller, has_wife, x), (x, has_na..."
4,0,5,Title 1 : Cadmium chloride \nParagraph 1 : Cad...,Cadmium Chloride is slightly soluble in this c...,alcohol,### CONTEXT_REASONING\nEntities were extracted...,Entities were extracted from each title and pa...,"[(cadmium_chloride, is_a, chemical_compound), ...","### QUESTION_REASONING\nFirst, identify entiti...","First, identify entities: ""Cadmium Chloride"" a...","[(cadmium_chloride, is_slightly_soluble_in, x)..."
5,0,6,Title 1 : Li Na \nParagraph 1 : Li Na (; ; bor...,Which tennis player won more Grand Slam titles...,Jonathan Stark,### CONTEXT_REASONING\nEntities were identifie...,Entities were identified as prominent individu...,"[(li_na, born_on, n26_february_1982), (li_na, ...","### QUESTION_REASONING\nFirst, identify the tw...","First, identify the two entities being compare...","[(henri_leconte, won_number_of_grand_slam_titl..."
6,0,7,"Title 1 : India \nParagraph 1 : India, officia...",Which genus of moth in the world's seventh-lar...,Crambidae,### CONTEXT_REASONING\nEntities were extracted...,Entities were extracted by identifying country...,"[(india, official_name, republic_of_india), (i...",### QUESTION_REASONING\n1. The question asks f...,1. The question asks for a genus of moth (unkn...,"[(x, is_a_genus_of, moth), (x, located_in, y),..."
7,0,8,Title 1 : Verano de Escándalo (1998) \nParagra...,Who was once considered the best kick boxer in...,Badr Hari,"### CONTEXT_REASONING\nFirst, I identified uni...","First, I identified unique entities from each ...","[(verano_de_escándalo_1998, held_on, september...","### QUESTION_REASONING\nFirst, I identify the ...","First, I identify the main subject of the ques...","[(x, was_considered_the_best_in, kick_boxing),..."
8,0,9,Title 1 : House of Anubis \nParagraph 1 : Hous...,"The Dutch-Belgian television series that ""Hous...",2006,### CONTEXT_REASONING\nI identified television...,"I identified television series, people, organi...","[(house_of_anubis, is_a, television_series), (...","### QUESTION_REASONING\nFirst, identify that ""...","First, identify that ""House of Anubis"" is base...","[(house_of_anubis, is_based_on, x), (x, first_..."
9,0,10,Title 1 : Mount 

## 5. Extracting sub-graphs from KGs

In [32]:
import hashlib
import math
import threading
from queue import Queue
from typing import Iterable, List, Tuple, Set

import psycopg2
from psycopg2.pool import SimpleConnectionPool
from psycopg2 import extensions as _pg_ext
from psycopg2 import sql
from pgvector.psycopg2 import register_vector

from rdflib import Graph, URIRef, BNode, Literal
from langchain_openai import OpenAIEmbeddings

# ------------------------------
# Configuration (DB EXACTLY as provided)
# ------------------------------
db_config = {
    'dbname': 'langchain',
    'user': 'langchain',
    'password': 'langchain',
    'host': 'localhost',
    'port': '6024'
}

# ------------------------------
# Connection pool + helpers
# ------------------------------
_DB_POOL = None
_DB_POOL_LOCK = threading.Lock()


def _get_pool() -> SimpleConnectionPool:
    global _DB_POOL
    if _DB_POOL is None:
        with _DB_POOL_LOCK:
            if _DB_POOL is None:
                _DB_POOL = SimpleConnectionPool(
                    minconn=1,
                    maxconn=16,  # adjust if you increase thread count
                    **db_config
                )
    return _DB_POOL


def _get_conn():
    pool = _get_pool()
    conn = pool.getconn()
    try:
        # Register pgvector adapter once per connection
        register_vector(conn)
    except Exception:
        pass
    _reset_conn(conn)
    return conn


def _put_conn(conn):
    try:
        _reset_conn(conn)  # return clean
    finally:
        _get_pool().putconn(conn)


def _reset_conn(conn):
    """Ensure the connection is not stuck in a failed or open transaction."""
    try:
        if conn.get_transaction_status() != _pg_ext.TRANSACTION_STATUS_IDLE:
            conn.rollback()
    except Exception:
        try:
            conn.rollback()
        except Exception:
            pass


# ------------------------------
# Schema management
# ------------------------------
def ensure_schema(embedding_dim: int = 1536):
    """
    - CREATE EXTENSION on a short-lived autocommit connection
    - CREATE TABLE/INDEXES on a pooled connection in a transaction
    """
    # 1) CREATE EXTENSION on its own connection (autocommit)
    try:
        ext_conn = psycopg2.connect(**db_config)
        ext_conn.autocommit = True
        with ext_conn.cursor() as cur:
            try:
                cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
            except psycopg2.Error as e:
                print("[ensure_schema] WARNING: CREATE EXTENSION vector failed.")
                print("[ensure_schema] pgerror:\n", getattr(e, "pgerror", str(e)))
                # Not fatal if already installed or if perms are restricted
    finally:
        try:
            ext_conn.close()
        except Exception:
            pass

    # 2) CREATE TABLE / INDEXES via pooled connection
    conn = _get_conn()
    try:
        with conn:
            with conn.cursor() as cur:
                cur.execute(sql.SQL("""
                    CREATE TABLE IF NOT EXISTS embeddings_cache (
                        id              BIGSERIAL PRIMARY KEY,
                        hash_word       TEXT        NOT NULL,
                        word            TEXT        NOT NULL,
                        word_embedding  vector({dim}) NOT NULL,
                        created_at      TIMESTAMPTZ NOT NULL DEFAULT now(),
                        updated_at      TIMESTAMPTZ NOT NULL DEFAULT now(),
                        UNIQUE (hash_word),
                        UNIQUE (word)
                    );
                """).format(dim=sql.Literal(embedding_dim)))

                # Attempt to align dimension (noop if same). May fail if existing data incompatible.
                try:
                    cur.execute(sql.SQL("""
                        ALTER TABLE embeddings_cache
                        ALTER COLUMN word_embedding TYPE vector({dim});
                    """).format(dim=sql.Literal(embedding_dim)))
                except psycopg2.Error as e:
                    print("[ensure_schema] NOTE: Could not alter word_embedding dimension.")
                    print("[ensure_schema] pgerror:\n", getattr(e, "pgerror", str(e)))

                cur.execute("""
                    CREATE INDEX IF NOT EXISTS idx_embeddings_cache_hash_word
                    ON embeddings_cache (hash_word);
                """)
    except psycopg2.Error as e:
        try:
            conn.rollback()
        except Exception:
            pass
        print("[ensure_schema] ERROR: DDL transaction failed and was rolled back.")
        print("[ensure_schema] pgerror:\n", getattr(e, "pgerror", str(e)))
        raise
    finally:
        _put_conn(conn)


# ---------- HASH HELPER ----------
def sha256_hex(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()


# ------------------------------
# Cache ops (DB)
# ------------------------------
def fetch_embedding_from_db(word: str):
    """
    Returns list[float] if present, else None.
    """
    h = sha256_hex(word)
    conn = _get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute(
                "SELECT word_embedding FROM embeddings_cache WHERE hash_word = %s",
                (h,)
            )
            row = cur.fetchone()
            return list(row[0]) if row else None
    finally:
        _put_conn(conn)


def upsert_embedding_in_db(word: str, embedding: List[float]):
    h = sha256_hex(word)
    conn = _get_conn()
    try:
        with conn:
            with conn.cursor() as cur:
                cur.execute(
                    """
                    INSERT INTO embeddings_cache (hash_word, word, word_embedding)
                    VALUES (%s, %s, %s)
                    ON CONFLICT (hash_word) DO UPDATE
                    SET word = EXCLUDED.word,
                        word_embedding = EXCLUDED.word_embedding,
                        updated_at = now();
                    """,
                    (h, word, embedding)
                )
    finally:
        _put_conn(conn)

In [33]:
import threading
import numpy as np
from queue import Queue
from langchain_openai import OpenAIEmbeddings
import ast


def short_name(uri):
    s = str(uri)
    if '#' in s:
        return s.split('#')[-1]
    elif '/' in s:
        return s.split('/')[-1]
    return s


# def load_embedding_dict(csv_path):
#     embedding_dict = {}
#     if os.path.exists(csv_path):
#         print(f"Loading existing embeddings from '{csv_path}'...")
#         df = pd.read_csv(csv_path)
#         for _, row in df.iterrows():
#             emb = ast.literal_eval(row["embedding"]) if isinstance(row["embedding"], str) else row["embedding"]
#             embedding_dict[row["string"]] = emb
#     else:
#         print(f"No embedding cache found. Initializing '{csv_path}' as empty.")
#         pd.DataFrame(columns=["string", "embedding"]).to_csv(csv_path, index=False)
#     return embedding_dict

def get_or_create_embedding(word: str, embedder: OpenAIEmbeddings) -> List[float]:
    cached = fetch_embedding_from_db(word)
    if cached is not None:
        return cached
    emb = embedder.embed_query(clean_entity(word))
    upsert_embedding_in_db(word, emb)
    return emb

# def get_embedding(text, embedding_dict, lock, embedder):
#     with lock:
#         if text in embedding_dict:
#             print(f"Embedding for '{text}' found in cache.")
#             return embedding_dict[text]
#     print(f"Requesting embedding for '{text}'...")
#     emb = embedder.embed_query(text)
#     with lock:
#         embedding_dict[text] = emb
#     print(f"Embedding for '{text}' cached.")
#     return emb

def cosine_similarity(vec1, vec2):
    v1 = np.array(vec1)
    v2 = np.array(vec2)
    if np.linalg.norm(v1) == 0 or np.linalg.norm(v2) == 0:
        return 0.0
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))


def bfs_n_hop_triples(graph, start_entity, n):
    visited = set([start_entity])
    current_level = set([start_entity])
    neighborhood_triples = set()
    for hop in range(n):
        next_level = set()
        for node in current_level:
            # Outgoing
            for s, p, o in graph.triples((node, None, None)):
                next_level.add(o)
                neighborhood_triples.add((short_name(s), short_name(p), short_name(o)))
            # Incoming
            for s, p, o in graph.triples((None, None, node)):
                next_level.add(s)
                neighborhood_triples.add((short_name(s), short_name(p), short_name(o)))
        next_level -= visited
        if not next_level:
            break
        visited |= next_level
        current_level = next_level
    return list(neighborhood_triples)


def extract_entities(graph):
    entities = set()
    for s, p, o in graph:
        # s = s.replace('_', ' ')
        # o = o.replace('_', ' ')
        # if s.startswith('n') and s[1].isdigit():
        #     s = s[1:]
        # if o.startswith('n') and o[1].isdigit():
        #     o = o[1:]
        entities.add(s)
        entities.add(o)
    return list(entities)


def clean_entity(entity):
    """
    Cleans the entity string by removing unwanted characters and formatting.
    """
    entity = entity.replace('_', ' ')
    if entity.startswith('n') and entity[1].isdigit():
        entity = entity[1:]
    return entity.strip().lower()


def worker(entity_queue,
           graph,
           key,
           key_embedding,
           per,
           n,
           neighborhoods,
           matched_entities,
           embedder,
           thread_id: int = 0,  # <-- default so it won't crash if omitted
           lock: threading.Lock = None):
    print(f"[Thread {thread_id}] started.")
    while not entity_queue.empty():
        entity = entity_queue.get()
        try:
            # entity_label = short_name(entity)
            entity_label = entity
            # emb = get_embedding(entity_label, embedding_dict, lock, embedder)
            emb = get_or_create_embedding(short_name(entity), embedder)
            sim = cosine_similarity(key_embedding, emb)
            if sim >= per:
                print(f"similarity for '{key}' and '{short_name(entity)}' is {sim:.4f}")
                print(f"Extracting {n}-hop triples for entity '{entity}'")
                triples = bfs_n_hop_triples(graph, entity, n)
                with lock:
                    neighborhoods.extend(triples)
                    matched_entities.append(entity_label)
        except Exception as e:
            print(f"[Thread {thread_id}] Error processing entity {entity}: {e}")
        finally:
            entity_queue.task_done()
    print(f"[Thread {thread_id}] finished.")

# def kg_neighborhood_extractor(
#     rdf_path, per, n, key, k=4, embedding_model="text-embedding-3-small", embedding_csv_path="embedding_dict.csv"
# ):
#     print("Loading RDF graph...")
#     graph = Graph()
#     graph.parse(rdf_path)
#     print("Extracting entities...")
#     entities = extract_entities(graph)
#     print(f"Total entities found: {len(entities)}")
#
#     embedding_dict = load_embedding_dict(embedding_csv_path)
#     neighborhoods = []
#     matched_entities = []
#     lock = threading.Lock()
#
#     print(f"Initializing LangChain OpenAIEmbeddings with model '{embedding_model}'...")
#     embedder = OpenAIEmbeddings(model=embedding_model)
#
#     print(f"Getting embedding for search key: '{key}'")
#     key_embedding = get_embedding(key, embedding_dict, lock, embedder)
#
#     entity_queue = Queue()
#     for entity in entities:
#         entity_queue.put(entity)
#
#     threads = []
#     for i in range(k):
#         t = threading.Thread(
#             target=worker,
#             args=(
#                 entity_queue,
#                 graph,
#                 key_embedding,
#                 per,
#                 n,
#                 embedding_dict,
#                 lock,
#                 neighborhoods,
#                 matched_entities,
#                 embedder,
#                 i+1,
#             ),
#         )
#         t.start()
#         threads.append(t)
#
#     entity_queue.join()
#     for t in threads:
#         t.join()
#
#     # Deduplicate triples
#     neighborhoods = list({triple for triple in neighborhoods})
#
#     # Save embedding_dict to CSV
#     with lock:
#         rows = [
#             {"string": s, "embedding": ",".join(map(str, v))}
#             for s, v in embedding_dict.items()
#         ]
#         df = pd.DataFrame(rows)
#         df.to_csv("embedding_dict.csv", index=False)
#
#     print("\nSummary:")
#     print(f"Matched entities: {matched_entities}")
#     print(f"Total unique triples in neighborhoods: {len(neighborhoods)}")
#     print("Embeddings saved as 'embedding_dict.csv'")
#     return neighborhoods

In [34]:
# ---------- MAIN FUNCTION (DB-backed cache) ----------
def kg_neighborhood_extractor(
        rdf_path,
        per,
        n,
        key,
        k=4,
        embedding_model="text-embedding-3-small",
        embedding_dim=1536  # keep in sync with the model you use
):
    """
    Args:
        rdf_path: path to RDF/XML/Turtle/etc. file supported by rdflib
        per: cosine similarity threshold (0..1) to treat an entity as a match
        n: number of BFS hops for neighborhood expansion
        key: search string to embed & compare with entity labels
        k: number of worker threads
        embedding_model: OpenAI embedding model name (via LangChain)
        embedding_dim: pgvector column dimension; must match the model in use
    """
    print("Ensuring DB schema...")
    ensure_schema(embedding_dim=embedding_dim)

    print("Loading RDF graph...")
    graph = Graph()
    graph.parse(rdf_path)

    print("Extracting entities...")
    entities = extract_entities(graph)
    print(f"Total entities found: {len(entities)}")

    neighborhoods: List[Tuple[str, str, str]] = []
    matched_entities: List[Tuple[str, float]] = []
    lock = threading.Lock()

    print(f"Initializing LangChain OpenAIEmbeddings with model '{embedding_model}'...")
    embedder = OpenAIEmbeddings(model=embedding_model)

    print(f"Getting embedding for search key: '{key}'")
    key_embedding = get_or_create_embedding(key, embedder)

    # Queue up work
    entity_queue = Queue()
    for ent in entities:
        entity_queue.put(ent)

    # Threads
    threads = []
    for i in range(k):
        t = threading.Thread(
            target=worker,
            args=(
                entity_queue,
                graph,
                key,
                key_embedding,
                per,
                n,
                neighborhoods,
                matched_entities,
                embedder,
                i + 1,  # <-- thread_id
                lock,  # <-- lock
            ),
            daemon=True,
        )
        t.start()
        threads.append(t)

    # Wait
    entity_queue.join()
    for t in threads:
        t.join()

    # Deduplicate triples
    neighborhoods = list({tr for tr in neighborhoods})

    # Optional: show cache size
    cached_count = None
    conn = _get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute("SELECT COUNT(*) FROM embeddings_cache;")
            cached_count = cur.fetchone()[0]
    finally:
        _put_conn(conn)

    print("\nSummary:")
    print(f"Matched entities (label, similarity): {matched_entities}")
    print(f"Total unique triples in neighborhoods: {len(neighborhoods)}")
    if cached_count is not None:
        print(f"Embeddings cached in DB: {cached_count}")

    return neighborhoods

In [35]:
# neighborhoods = kg_neighborhood_extractor(
#     rdf_path="outputs/knowledge_graphs/hotpotQA_ontology_b0_q1.rdf",
#     per=0.75, n=2, key="arthurs_magazine", k=3, embedding_model="text-embedding-3-small"
# )
# print("\nNeighborhood triples:")
# for triple in neighborhoods:
#     print(triple)

In [36]:
# loop through each row in main_df and extract neighborhoods
batch_input_list = []

for i in range(0, main_df.shape[0], BATCH_SIZE):
    batch_df = main_df.iloc[i:i + BATCH_SIZE]
    batch_input_item = []

    for i, (index, row) in enumerate(batch_df.iterrows()):
        reference_triples = []
        for triple in row['question_triples']:
            # Extract reference_triples using question_triples
            if len(triple[0]) > 1:
                reference_triples += kg_neighborhood_extractor(
                    rdf_path=f"{KNOWLEDGE_GRAPH_PATH}/{DATASET_NAME}_ontology_b{row['batch_num']}_q{row['question_num']}.rdf",
                    per=0.75, n=3, key=triple[0], k=3, embedding_model="text-embedding-3-small"
                )
            print("-" * 50)
            if len(triple[2]) > 1:
                reference_triples += kg_neighborhood_extractor(
                    rdf_path=f"{KNOWLEDGE_GRAPH_PATH}/{DATASET_NAME}_ontology_b{row['batch_num']}_q{row['question_num']}.rdf",
                    per=0.75, n=3, key=triple[2], k=3, embedding_model="text-embedding-3-small"
                )

        batch_input_item.append({
            "question": row['question'],
            "reference_triples": reference_triples
        })
    print("=" * 50)
    batch_input_list.append(batch_input_item)
    break

Ensuring DB schema...
Loading RDF graph...
Extracting entities...
Total entities found: 145
Initializing LangChain OpenAIEmbeddings with model 'text-embedding-3-small'...
Getting embedding for search key: 'arthurs_magazine'
[Thread 1] started.
[Thread 2] started.
[Thread 3] started.
similarity for 'arthurs_magazine' and 'arthurs_magazine' is 1.0000
Extracting 3-hop triples for entity 'http://example.org/arthurs_magazine'
[Thread 2] finished.
[Thread 3] finished.
[Thread 1] finished.

Summary:
Matched entities (label, similarity): [rdflib.term.URIRef('http://example.org/arthurs_magazine')]
Total unique triples in neighborhoods: 9
Embeddings cached in DB: 1331
--------------------------------------------------
Ensuring DB schema...
Loading RDF graph...
Extracting entities...
Total entities found: 145
Initializing LangChain OpenAIEmbeddings with model 'text-embedding-3-small'...
Getting embedding for search key: 'first_for_women'
[Thread 1] started.
[Thread 2] started.
[Thread 3] started.

In [37]:
# convert '_' into ' ' and remove first 'n' if second charactor is an integer of every entity or relationship of reference_triples
def format_reference_triples(reference_triples):
    formatted_triples = []
    for triple in reference_triples:
        subject, relation, obj = triple
        subject = subject.replace('_', ' ')
        relation = relation.replace('_', ' ')
        obj = obj.replace('_', ' ')
        if subject.startswith('n') and subject[1].isdigit():
            subject = subject[1:]
        if obj.startswith('n') and obj[1].isdigit():
            obj = obj[1:]
        formatted_triples.append((subject, relation, obj))
    return formatted_triples


for i, batch in enumerate(batch_input_list):
    print(f"Batch {i} ... processing {len(batch)} items")
    for j, item in enumerate(batch):
        item['reference_triples'] = format_reference_triples(item['reference_triples'])
        print(f"Item {j}:")
        print("Question:", item['question'])
        print("Reference Triples:", item['reference_triples'])
        print("-" * 50)

Batch 0 ... processing 10 items
Item 0:
Question: Which magazine was started first Arthur's Magazine or First for Women?
Reference Triples: [('arthurs magazine', 'published in', 'philadelphia'), ('arthurs magazine', 'is', 'american literary periodical'), ('t.s. arthur', 'editor of', 'arthurs magazine'), ('sarah josepha hale', 'featured in', 'arthurs magazine'), ('arthurs magazine', 'published during', '1844 1846'), ('thomas g. spear', 'featured in', 'arthurs magazine'), ('arthurs magazine', 'merged into', 'godeys ladys book'), ('j.h. ingraham', 'featured in', 'arthurs magazine'), ('edgar a. poe', 'featured in', 'arthurs magazine'), ('first for women', 'published in', 'usa'), ('womens colleges in the southern united states', 'origin', 'started as girls seminaries or academies'), ('womens colleges in the southern united states', 'located in', 'southern united states'), ('first for women', 'based in', 'englewood cliffs new jersey'), ('first for women', 'circulation', '1310696 copies in 20

## 6. Getting the final answer

In [38]:
system_msg = """
<role>
    You are a logical reasoning expert tasked with answering questions based only on provided structured knowledge in the form of Reference Triples.
</role>

<behavior>
    <rule name="Step-by-Step Reasoning">
        Use the reference triples to deduce relevant facts in a step-by-step manner. Clearly identify which triples are used for each inference.
    </rule>
    <rule name="Comparative Reasoning">
        When comparing entities, extract or infer comparable attributes (e.g., start dates, founding years) and use them to conclude which comes first, is larger, etc., as applicable.
    </rule>
    <rule name="Answer Reporting">
        At the end of your reasoning, provide the final answer on a new line prefixed with "### FINAL_ANSWER", followed by the concise answer on the next line.
        If the data is insufficient to answer the question definitively, return "insufficient data".
    </rule>
</behavior>

<format>
1. Carefully examine the <question> and <reference_triples>.
2. Begin with step-by-step reasoning based strictly on the reference triples.
3. End your output with the heading "### FINAL_ANSWER" followed by the answer on the next line.
</format>
"""

human_msg = """
<question>
{question}
</question>

<reference triples>
{reference_triples}
</reference triples>
"""

prompt = ChatPromptTemplate([("system", system_msg), ("human", human_msg)])

In [39]:
chain = prompt | llm_model | StrOutputParser()

In [40]:
answer_str_batch = []

for i, batch in enumerate(batch_input_list):
    print(f"Batch {i + 1} ... processing {len(batch)} items")
    answer_str_list = chain.batch(batch)
    answer_str_batch.append(answer_str_list)

Batch 1 ... processing 10 items


In [41]:
for i, answer_str_list in enumerate(answer_str_batch):
    print(f"Batch {i + 1} ... processed {len(answer_str_list)} items")
    for j, answer_str in enumerate(answer_str_list):
        print(f"Item {j + 1}:")
        print(answer_str)
        print("-" * 50)

# Arthur's Magazine
# Delhi
# President Richard Nixon
# American
# alcohol
# Jonathan Stark
# Crambidae
# Badr Hari
# 2006
# 6.213 km long

Batch 1 ... processed 10 items
Item 1:
Step-by-Step Reasoning:

1. The question asks which magazine was started first: Arthur's Magazine or First for Women.
2. Locate relevant start/publish information for each magazine in the reference triples.

- For "arthurs magazine":
    - Triple: ('arthurs magazine', 'published during', '1844 1846')
      This indicates Arthur's Magazine was active during 1844 to 1846, so it started no later than 1844.
- For "first for women":
    - Triple: ('first for women', 'started in', '1989')
      This indicates First for Women started in 1989.

3. Now, compare the starting years:
    - Arthur's Magazine: 1844
    - First for Women: 1989

4. 1844 comes before 1989.

### FINAL_ANSWER
Arthur's Magazine was started first.
--------------------------------------------------
Item 2:
Step-by-step reasoning:

1. The question asks about the city where the head office is located for the hotel company associated with the Oberoi family.
2. From the triple ('oberoi fam

In [42]:
llm_answer_str_df = pd.DataFrame({
    "batch_num": [],
    "question_num": [],
    "llm_answer_str": []
})

for i, answer_str_list in enumerate(answer_str_batch):
    batch_num_list = [i] * len(answer_str_list)
    question_num_list = list(range(1, len(answer_str_list) + 1))
    llm_answer_str_df = pd.concat([llm_answer_str_df, pd.DataFrame({
        "batch_num": batch_num_list,
        "question_num": question_num_list,
        "llm_answer_str": answer_str_list
    })], ignore_index=True)

llm_answer_str_df

,batch_num,question_num,llm_answer_str
0,0.0,1.0,Step-by-Step Reasoning:\n\n1. The question ask...
1,0.0,2.0,Step-by-step reasoning:\n\n1. The question ask...
2,0.0,3.0,Step-by-step reasoning:\n\n1. The question ask...
3,0.0,4.0,Step-by-step reasoning:\n\n1. The question ask...
4,0.0,5.0,Step-by-step reasoning:\n\n1. The question ask...
5,0.0,6.0,Step-by-step reasoning:\n\n1. The question ask...
6,0.0,7.0,Step-by-step reasoning:\n\n1. The question ask...
7,0.0,8.0,Step-by-step reasoning:\n\n1. The question ask...
8,0.0,9.0,Step-by-step reasoning:\n\n1. The question ask...
9,0.0,10.0,Step-by-step reasoning:\n\n1. The question ask...


In [43]:
# Extract the final answer from the answer_str

def extract_final_answer(row):
    # Regex pattern to find the final answer
    final_answer_pat = r'### FINAL_ANSWER\s*(.*)'
    match = re.search(final_answer_pat, row['llm_answer_str'], re.DOTALL)
    return match.group(1).strip() if match else "No answer found"


llm_answer_str_df['llm_answer'] = llm_answer_str_df.apply(extract_final_answer, axis=1)
llm_answer_str_df

,batch_num,question_num,llm_answer_str,llm_answer
0,0.0,1.0,Step-by-Step Reasoning:\n\n1. The question ask...,Arthur's Magazine was started first.
1,0.0,2.0,Step-by-step reasoning:\n\n1. The question ask...,Delhi
2,0.0,3.0,Step-by-step reasoning:\n\n1. The question ask...,Richard Nixon's middle name
3,0.0,4.0,Step-by-step reasoning:\n\n1. The question ask...,insufficient data
4,0.0,5.0,Step-by-step reasoning:\n\n1. The question ask...,ethyl alcohol (ethanol)
5,0.0,6.0,Step-by-step reasoning:\n\n1. The question ask...,Jonathan Stark
6,0.0,7.0,Step-by-step reasoning:\n\n1. The question ask...,insufficient data
7,0.0,8.0,Step-by-step reasoning:\n\n1. The question ask...,Badr Hari
8,0.0,9.0,Step-by-step reasoning:\n\n1. The question ask...,September 2006
9,0.0,10.0,Step-by-step reasoning:\n\n1. The question ask...,6.213 km


In [44]:
# Merge the llm_answer_str_df with main_df to get the final answers
main_df = main_df.merge(llm_answer_str_df, on=['batch_num', 'question_num'], how='left')
del llm_answer_str_df

In [45]:
main_df[['question', 'batch_num', 'question_num', 'answer', 'llm_answer']]

,question,batch_num,question_num,answer,llm_answer
0,Which magazine was started first Arthur's Maga...,0,1,Arthur's Magazine,Arthur's Magazine was started first.
1,The Oberoi family is part of a hotel company t...,0,2,Delhi,Delhi
2,Musician and satirist Allie Goertz wrote a son...,0,3,President Richard Nixon,Richard Nixon's middle name
3,What nationality was James Henry Miller's wife?,0,4,American,insufficient data
4,Cadmium Chloride is slightly soluble in this c...,0,5,alcohol,ethyl alcohol (ethanol)
5,Which tennis player won more Grand Slam titles...,0,6,Jonathan Stark,Jonathan Stark
6,Which genus of moth in the world's seventh-lar...,0,7,Crambidae,insufficient data
7,Who was once considered the best kick boxer in...,0,8,Badr Hari,Badr Hari
8,"The Dutch-Belgian television series that ""Hous...",0,9,2006,September 2006
9,What is the length of the track where the 2013...,0,10,6.213 km long,6.213 km


## 7. Calculating the accuracy

In [46]:
system_msg = """
<role>
    You are a meticulous LLM answer evaluator. Your task is to determine if the provided llm_answer correctly matches the correct_answer for a given question.
</role>

<behavior>
    <rule name="case-insensitive-match">
        Treat answers as correct even if their case (uppercase/lowercase) does not match, as long as the content matches.
    </rule>
    <rule name="synonyms-acceptable">
        Accept synonyms, short forms, or equivalent factual answers (e.g., "alcohol" and "ethanol") as correct, unless there is a clear difference in meaning or context.
    </rule>
    <rule name="factual-containment">
        Accept answers that contain the correct answer as a substring, or are paraphrased, provided no contradictory information is introduced.
    </rule>
    <rule name="insufficient-data">
        Mark as incorrect if llm_answer indicates "insufficient data" or similar phrases, but a correct factual answer is actually provided in correct_answer.
    </rule>
    <rule name="incorrect-content">
        Mark as incorrect if the llm_answer gives a wrong, contradictory, or irrelevant response compared to correct_answer.
    </rule>
    <rule name="format-neutrality">
        Do not penalize for minor differences in format, such as punctuation or extra explanatory words, as long as the meaning is unchanged.
    </rule>
    <rule name="numerical-tolerance">
        For numerical answers, accept equivalent formats (e.g., "2006" and "September 2006" are correct if both indicate the correct year), but mark as incorrect if the core value is wrong.
    </rule>
</behavior>

<format>
1) Output your reasoning step by step, referencing the question, correct_answer, and llm_answer.
2) Clearly state if the llm_answer matches the correct_answer according to the behavior rules above.
3) End your output with the heading "### FINAL_ANSWER" followed by TRUE if the llm_answer is correct or FALSE if incorrect.
</format>
"""

human_msg = """
<question>
{question}
</question>

<correct_answer>
{correct_answer}
</correct_answer>

<llm_answer>
{llm_answer}
</llm_answer>
"""

prompt = ChatPromptTemplate([("system", system_msg), ("human", human_msg)])

In [47]:
chain = prompt | llm_model | StrOutputParser()

In [48]:
# preparing the input for the evaluation chain
batch_input_list = []

for i in range(0, main_df.shape[0], BATCH_SIZE):
    batch_df = main_df.iloc[i:i + BATCH_SIZE]
    batch_input_item = []

    for i, (index, row) in enumerate(batch_df.iterrows()):
        batch_input_item.append({
            "question": row['question'],
            "correct_answer": row['answer'],
            "llm_answer": row['llm_answer']
        })
    batch_input_list.append(batch_input_item)

for i, batch in enumerate(batch_input_list):
    print(f"Batch {i + 1}:")
    for item in batch:
        print("Question:", item['question'])
        print("Correct Answer:", item['correct_answer'])
        print("LLM Answer:", item['llm_answer'])
        print("-" * 50)

Batch 1:
Question: Which magazine was started first Arthur's Magazine or First for Women?
Correct Answer: Arthur's Magazine
LLM Answer: Arthur's Magazine was started first.
--------------------------------------------------
Question: The Oberoi family is part of a hotel company that has a head office in what city?
Correct Answer: Delhi
LLM Answer: Delhi
--------------------------------------------------
Question: Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?
Correct Answer: President Richard Nixon
LLM Answer: Richard Nixon's middle name
--------------------------------------------------
Question:  What nationality was James Henry Miller's wife?
Correct Answer: American
LLM Answer: insufficient data
--------------------------------------------------
Question: Cadmium Chloride is slightly soluble in this chemical, it is also called what?
Correct Answer: alcohol
LLM Answer: ethyl alcohol (ethanol)
----------

In [49]:
answer_check_str_batch = []

for i, batch in enumerate(batch_input_list):
    print(f"Batch {i + 1} ... processing {len(batch)} items")
    answer_check_str_list = chain.batch(batch)
    answer_check_str_batch.append(answer_check_str_list)

Batch 1 ... processing 10 items


In [50]:
for i, answer_check_str_list in enumerate(answer_check_str_batch):
    print(f"Batch {i + 1} ... processed {len(answer_check_str_list)} items")
    for j, answer_check_str in enumerate(answer_check_str_list):
        print(f"Item {j + 1}:")
        print(answer_check_str)
        print("-" * 50)

Batch 1 ... processed 10 items
Item 1:
1) Step-by-step reasoning:
- The question is: Which magazine was started first Arthur's Magazine or First for Women?
- The correct_answer is: Arthur's Magazine
- The llm_answer is: Arthur's Magazine was started first.

- The llm_answer directly answers the question in a clear, affirmative sentence.
- It states that "Arthur's Magazine was started first," which is exactly the information requested and matches the correct_answer.
- According to the rules:
    - case-insensitive-match: Both answers refer to "Arthur's Magazine," case is not an issue.
    - factual-containment: The llm_answer restates the correct fact and does not add contradictory or irrelevant information.
    - format-neutrality: The answer is slightly rephrased but the meaning and content are fully correct.

2) The llm_answer matches the correct_answer according to all provided behavior rules.

### FINAL_ANSWER
TRUE
--------------------------------------------------
Item 2:
1) Let's

In [51]:
# Create a DataFrame to store the evaluation results
answer_check_df = pd.DataFrame({
    "batch_num": [],
    "question_num": [],
    "answer_check_str": []
})

for i, answer_check_str_list in enumerate(answer_check_str_batch):
    batch_num_list = [i] * len(answer_check_str_list)
    question_num_list = list(range(1, len(answer_check_str_list) + 1))
    answer_check_df = pd.concat([answer_check_df, pd.DataFrame({
        "batch_num": batch_num_list,
        "question_num": question_num_list,
        "answer_check_str": answer_check_str_list
    })], ignore_index=True)

In [52]:
answer_check_df

,batch_num,question_num,answer_check_str
0,0.0,1.0,1) Step-by-step reasoning:\n- The question is:...
1,0.0,2.0,"1) Let's evaluate the question, correct_answer..."
2,0.0,3.0,1) Reasoning step by step:\n- The question ask...
3,0.0,4.0,1) Reasoning step by step:\n\n- The question a...
4,0.0,5.0,"1) Let's analyze the question, which asks abou..."
5,0.0,6.0,"1) The question asks which tennis player, betw..."
6,0.0,7.0,1) Step-by-step reasoning:\n- The question ask...
7,0.0,8.0,1) Reasoning:\n- The question asks for the nam...
8,0.0,9.0,1) Reasoning step-by-step:\n- The question ask...
9,0.0,10.0,1) Reasoning step by step:\n- The question ask...


In [53]:
# Extract is_correct from the answer_check_str
def extract_is_correct(row):
    # Regex pattern to find the final answer
    final_answer_pat = r'### FINAL_ANSWER\s*(TRUE|FALSE)'
    match = re.search(final_answer_pat, row['answer_check_str'], re.DOTALL)
    return match.group(1).strip() if match else False


# Merge the llm_answer_str_df with main_df to get the final answers
answer_check_df['is_correct'] = answer_check_df.apply(extract_is_correct, axis=1)
main_df = main_df.merge(answer_check_df, on=['batch_num', 'question_num'], how='left')
main_df

,batch_num,question_num,context,question,answer,graphs_str,context_reasoning,context_triples,questions_str,question_reasoning,question_triples,llm_answer_str,llm_answer,answer_check_str,is_correct
0,0,1,Title 1 : Radio City (Indian radio station) \n...,Which magazine was started first Arthur's Maga...,Arthur's Magazine,### CONTEXT_REASONING\nEntities were extracted...,Entities were extracted by identifying names o...,"[(radio_city, is, radio_station), (radio_city,...",### QUESTION_REASONING\nI identified two entit...,"I identified two entities in the question: ""Ar...","[(arthurs_magazine, started_in, x), (first_for...",Step-by-Step Reasoning:\n\n1. The question ask...,Arthur's Magazine was started first.,1) Step-by-step reasoning:\n- The question is:...,TRUE
1,0,2,Title 1 : Ritz-Carlton Jakarta \nParagraph 1 :...,The Oberoi family is part of a hotel company t...,Delhi,### CONTEXT_REASONING\nEntities were extracted...,Entities were extracted based on clear mention...,"[(ritz-carlton_jakarta, is_a, hotel), (ritz-ca...",### QUESTION_REASONING\nThe question is asking...,The question is asking for the city where the ...,"[(oberoi_family, is_part_of, x), (x, has_head_...",Step-by-step reasoning:\n\n1. The question ask...,Delhi,"1) Let's evaluate the question, correct_answer...",TRUE
2,0,3,Title 1 : Lisa Simpson \nParagraph 1 : Lisa Ma...,Musician and satirist Allie Goertz wrote a son...,President Richard Nixon,### CONTEXT_REASONING\nEntities are extracted ...,Entities are extracted by identifying unique n...,"[(lisa_simpson, is_fictional_character_in, the...","### QUESTION_REASONING\nFirst, identify the ma...","First, identify the main entities: Allie Goert...","[(allie_goertz, wrote_a_song_about, milhouse),...",Step-by-step reasoning:\n\n1. The question ask...,Richard Nixon's middle name,1) Reasoning step by step:\n- The question ask...,TRUE
3,0,4,"Title 1 : Moloch: or, This Gentile World \nPar...",What nationality was James Henry Miller's wife?,American,### CONTEXT_REASONING\nEntities were extracted...,Entities were extracted from titles and paragr...,"[(moloch_or, this_gentile_world, written_by_he...",### QUESTION_REASONING\nThe question asks for ...,The question asks for the nationality of James...,"[(james_henry_miller, has_wife, x), (x, has_na...",Step-by-step reasoning:\n\n1. The question ask...,insufficient data,1) Reasoning step by step:\n\n- The question a...,FALSE
4,0,5,Title 1 : Cadmium chloride \nParagraph 1 : Cad...,Cadmium Chloride is slightly soluble in this c...,alcohol,### CONTEXT_REASONING\nEntities were extracted...,Entities were extracted from each title and pa...,"[(cadmium_chloride, is_a, chemical_compound), ...","### QUESTION_REASONING\nFirst, identify entiti...","First, identify entities: ""Cadmium Chloride"" a...","[(cadmium_chloride, is_slightly_soluble_in, x)...",Step-by-step reasoning:\n\n1. The question ask...,ethyl alcohol (ethanol),"1) Let's analyze the question, which asks abou...",TRUE
5,0,6,Title 1 : Li Na \nParagraph 1 : Li Na (; ; bor...,Which tennis player won more Grand Slam titles...,Jonathan Stark,### CONTEXT_REASONING\nEntities were identifie...,Entities were identified as prominent individu...,"[(li_na, born_on, n26_february_1982), (li_na, ...","### QUESTION_REASONING\nFirst, identify the tw...","First, identify the two entities being compare...","[(henri_leconte, won_number_of_grand_slam_titl...",Step-by-step reasoning:\n\n1. The question ask...,Jonathan Stark,"1) The question asks which tennis player, betw...",TRUE
6,0,7,"Title 1 : India \nParagraph 1 : India, officia...",Which genus of moth in the world's seventh-lar...,Crambidae,### CONTEXT_REASONING\nEntities were extracted...,Entities were extracted by identifying country...,"[(india, official_name, republic_of_india), (i...",### QUESTION_REASONING\n1. The question asks f...,1. The question asks for a genus of moth (unkn...,"[(x, is_a_genus_of, moth), (x, located_in, y),...",Step-by-step reasoning:\n\n1. The question ask...,insuffici

In [54]:
# Calculate the accuracy
accuracy = main_df['is_correct'].value_counts(normalize=True).get('TRUE', 0) * 100
print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 80.00%


In [55]:
main_df.to_csv("outputs/hotpotqa_complex_entity_prediction_results.csv", index=False)

In [56]:
# GPT 4.1 - Accuracy: 80.00%
# GPT 4.1-mini - Accuracy: 80.00%